In [3]:
!pip install ultralytics

In [4]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/KP/YASA_LOITERING DETECTION/v1.4.0"

Mounted at /content/drive
/content/drive/MyDrive/KP/YASA_LOITERING DETECTION/v1.4.0


In [5]:
import cv2
import torch
import time
from collections import defaultdict
from ultralytics import YOLO
import numpy as np
from tqdm import tqdm
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
def detect_persons_with_time_tracking(model_path, video_path, output_path, detection_threshold_seconds=5):
    try:
        model = YOLO(model_path)
    except Exception as e:
        print(f"Error loading YOLO model: {e}")
        return 0, 0  # No frames, no time

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return 0, 0

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    if not out.isOpened():
        cap.release()
        return 0, 0

    person_detection_start_times = {}
    person_last_bbox = {}

    frame_count = 0
    start_time = time.time()

    with tqdm(total=total_frames, desc=f"Processing {os.path.basename(video_path)}", unit="frame") as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            current_frame_time = time.time()

            results = model(frame, conf=0.5, classes=[0])  # Only detect persons (COCO class 0)

            current_frame_detected_persons = []
            for r in results:
                boxes = r.boxes
                for box in boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])
                    cls = int(box.cls[0])

                    if model.names[cls] == 'person':
                        x_center = (x1 + x2) / 2
                        y_center = (y1 + y2) / 2

                        matched_person_id = None
                        min_distance = float('inf')
                        for person_id in person_last_bbox:
                            last_x_center, last_y_center = person_last_bbox[person_id]
                            distance = np.sqrt((x_center - last_x_center)**2 + (y_center - last_y_center)**2)
                            if distance < 50 and distance < min_distance:
                                min_distance = distance
                                matched_person_id = person_id

                        if matched_person_id is None:
                            person_id = (frame_count, x_center, y_center)
                            person_detection_start_times[person_id] = current_frame_time
                            person_last_bbox[person_id] = (x_center, y_center)
                            current_frame_detected_persons.append((person_id, (x1, y1, x2, y2)))
                        else:
                            person_last_bbox[matched_person_id] = (x_center, y_center)
                            current_frame_detected_persons.append((matched_person_id, (x1, y1, x2, y2)))

            for person_id, (x1, y1, x2, y2) in current_frame_detected_persons:
                detection_start_time = person_detection_start_times.get(person_id, current_frame_time)
                detection_duration = current_frame_time - detection_start_time

                color = (255, 0, 0)
                label = f"Person ({detection_duration:.1f}s)"

                if detection_duration > detection_threshold_seconds:
                    color = (0, 0, 255)
                    label = f"Loiterer ({detection_duration:.1f}s)"

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            out.write(frame)
            pbar.update(1)

    cap.release()
    out.release()

    end_time = time.time()
    execution_time = end_time - start_time
    return frame_count, execution_time

In [7]:
if __name__ == "__main__":
    yolo_model_path = 'yolo12m.pt'
    total_frames_all_videos = 0
    total_time_all_videos = 0.0

    for i in tqdm(range(1, 11), desc="Processing videos", unit="video"):
        input_video_path = f'sample_{i}.mp4'
        output_video_path = f'output_{i}.mp4'

        if not os.path.exists(input_video_path):
            tqdm.write(f"[WARNING] {input_video_path} not found, skipping...")
            continue

        frames, exec_time = detect_persons_with_time_tracking(
            yolo_model_path,
            input_video_path,
            output_video_path,
            detection_threshold_seconds=3
        )

        total_frames_all_videos += frames
        total_time_all_videos += exec_time

    if total_frames_all_videos > 0:
        avg_time_per_frame = total_time_all_videos / total_frames_all_videos
        print(f"\n[RESULT] Average Time per Frame across all videos: {avg_time_per_frame:.4f} seconds")
    else:
        print("\n[RESULT] No frames processed.")

Processing videos:   0%|          | 0/10 [00:00<?, ?video/s]

Processing sample_1.mp4:   0%|          | 0/493 [00:00<?, ?frame/s]


0: 384x640 (no detections), 64.6ms
Speed: 17.8ms preprocess, 64.6ms inference, 145.9ms postprocess per image at shape (1, 3, 384, 640)



Processing sample_1.mp4:   0%|          | 1/493 [00:06<57:15,  6.98s/frame]


0: 384x640 (no detections), 27.7ms
Speed: 2.3ms preprocess, 27.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 27.7ms
Speed: 2.3ms preprocess, 27.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 27.7ms
Speed: 2.1ms preprocess, 27.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)



Processing sample_1.mp4:   1%|          | 4/493 [00:07<11:03,  1.36s/frame]

...


0: 384x640 (no detections), 22.8ms
Speed: 2.4ms preprocess, 22.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 23.9ms
Speed: 2.3ms preprocess, 23.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 25.9ms
Speed: 2.4ms preprocess, 25.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)



Processing sample_10.mp4:  99%|█████████▉| 191/192 [00:09<00:00, 22.63frame/s]


0: 384x640 1 person, 27.2ms
Speed: 2.3ms preprocess, 27.2ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)


Processing videos: 100%|██████████| 10/10 [02:15<00:00, 13.54s/video]


[RESULT] Average Time per Frame across all videos: 0.0518 seconds
